In [1]:
from typing import TypedDict

from langchain_openai import ChatOpenAI
from langgraph.graph import StateGraph, START, END
from langgraph.checkpoint.memory import InMemorySaver

d:\gen_ai\Gen-AI\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


In [2]:
llm = ChatOpenAI(model="gpt-4", temperature=0.7)

In [3]:
class jokeState(TypedDict):
    topic: str
    joke: str
    summary: str


def generate_joke(state: jokeState) -> jokeState:

    topic = state["topic"]
    joke_prompt = f"Generate a funny joke about {topic}."
    joke = llm.invoke(joke_prompt)
    return {"topic": topic, "joke": joke}

def generate_summary(state: jokeState) -> jokeState:
    joke = state["joke"]
    summary_prompt = f"Summarize the following joke in one sentence: {joke}"
    summary = llm.invoke(summary_prompt)
    return {"topic": state["topic"], "joke": joke, "summary": summary}


In [4]:
graph = StateGraph(jokeState)

# add nodes

graph.add_node("generate_joke", generate_joke)
graph.add_node("generate_summary", generate_summary)


# add edges

graph.add_edge(START, "generate_joke")
graph.add_edge("generate_joke", "generate_summary")
graph.add_edge("generate_summary", END)

# compile the graph with persistent
memory = InMemorySaver()
workflow = graph.compile(checkpointer=memory)

In [9]:
config = {"configurable": {"thread_id": "1"}}

result = workflow.invoke({"topic": "programming"}, config=config)
print(result["summary"])

content='The joke suggests that programmers dislike nature because it contains too many bugs, playing on the dual meaning of "bugs" as both insects and software errors.' additional_kwargs={'refusal': None} response_metadata={'token_usage': {'completion_tokens': 30, 'prompt_tokens': 306, 'total_tokens': 336, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cache_write_tokens': None, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4-0613', 'system_fingerprint': None, 'id': 'chatcmpl-ELSkGfcnsN7a93zPi8Eef2tvCpAt2', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None} id='lc_run--01a07bd7-247a-7011-96b6-b98844c75c21-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 306, 'output_tokens': 30, 'total_tokens': 336, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 